In [0]:
%skip
%pip install numba

In [0]:
dbutils.library.restartPython()

In [0]:
# COMMAND ----------
# Define Widgets for User Input
dbutils.widgets.text("snapshot", "202505", "Snapshot (YYYYMM)")
dbutils.widgets.text("DIL", "", "DIL Value (Optional)")
dbutils.widgets.text("product", "", "Product (Optional)")

# Retrieve Parameters
SNAPSHOT = dbutils.widgets.get("snapshot")
DIL = dbutils.widgets.get("DIL")
PRODUCT = dbutils.widgets.get("product")

# COMMAND ----------
import os
import sys
import logging
import yaml
from argparse import Namespace

# Add the repo's src directory to the Python path to allow imports
# This assumes the notebook is located in amfs_tm/src/
REPO_ROOT = os.path.abspath("/Workspace/Users/hanif.m.abidin@gmail.com/amfs_telemarketing_new_acquisition")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from lib.clean.clean_all import clean_all
from lib.feature.features_generation import features_generation
from lib.matrix.merge_all_features import create_modeling_matrix_pandas_dynamic
from lib.matrix.impute_dummy import impute_and_dummy_matrix

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# COMMAND ----------
# Define Configuration Paths
# Based on the migrated repository structure
CONFIG_DIR = os.path.join(REPO_ROOT, "configs")
MAIN_CONFIG = os.path.join(CONFIG_DIR, "main_config.yaml")
CLEAN_CONFIG = os.path.join(CONFIG_DIR, "clean_config.yaml")
FEATURES_CONFIG = os.path.join(CONFIG_DIR, "features_config.yaml")

# Create a mock args object to pass to the library functions
args = Namespace(snapshot=SNAPSHOT, DIL=DIL, product=PRODUCT)

In [0]:
# COMMAND ----------
# DBTITLE 1,Step 1: Data Cleaning
# Processes raw files into a standardized format
# import importlib
# from lib.clean import clean_config_utils, clean_utils, clean_all as clean_all_module
# importlib.reload(clean_config_utils)
# importlib.reload(clean_utils)
# importlib.reload(clean_all_module)
# from lib.clean.clean_all import clean_all

logging.info(f"Starting Data Cleaning for snapshot {SNAPSHOT}...")
# Note: Ensure paths in clean_config.yaml are updated to Volume paths
clean_all(MAIN_CONFIG, CLEAN_CONFIG, args)

In [0]:
# COMMAND ----------
# DBTITLE 1,Step 2: Feature Generation
logging.info(f"Starting Feature Generation...")
features_generation(MAIN_CONFIG, FEATURES_CONFIG, args)

In [0]:
# COMMAND ----------
# DBTITLE 1,Step 3: Matrix Creation
# Merges all features and the base population into a modeling matrix
logging.info(f"Merging features into modeling matrix...")
create_modeling_matrix_pandas_dynamic(MAIN_CONFIG, args)

In [0]:
# COMMAND ----------
# DBTITLE 1,Step 4: Imputation and Encoding
# Finalizes the matrix with missing value handling and dummy variables
logging.info(f"Applying imputation and dummification...")
impute_and_dummy_matrix(MAIN_CONFIG, args)

logging.info("Pipeline Execution Complete.")